# Multimodal Embedding with Qwen3-VL and OpenVINO

The [Qwen3-VL-Embedding model series](https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B) is built upon the powerful Qwen3-VL foundation model, specifically designed for multimodal embedding tasks. It accepts diverse inputs including text, images, screenshots, and videos, as well as inputs containing a mixture of these modalities. The model generates high-dimensional vectors for broad applications like retrieval, clustering, and classification.

<img src="https://model-demo.oss-cn-hangzhou.aliyuncs.com/Qwen3-VL-Embedding.png" width="500"/>

In this tutorial we consider how to convert Qwen3-VL Embedding model with [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) and run inference with [OpenVINO GenAI](https://github.com/openvinotoolkit/openvino.genai) `EmbeddingPipeline`.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Select model](#Select-model)
- [Convert model using Optimum Intel](#Convert-model-using-Optimum-Intel)
- [Run OpenVINO GenAI model inference](#Run-OpenVINO-GenAI-model-inference)


<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/qwen3-vl-embedding/qwen3-vl-embedding.ipynb" />

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [1]:
%pip uninstall -q -y optimum optimum-intel optimum-onnx
%pip install -q "optimum-intel[openvino] @ https://codeload.github.com/huggingface/optimum-intel/tar.gz/f48d93fddff8c91e198389c47a6d5974789b67f4" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q "transformers>=4.57,<=5.0" "torch>=2.9" "torchvision" "qwen-vl-utils>=0.0.14" "pillow" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -qU --pre "openvino" "openvino-tokenizers" "openvino-genai" --extra-index-url https://storage.openvinotoolkit.org/simple/wheels/nightly

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
from pathlib import Path

if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    open("cmd_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("qwen3-vl-embedding.ipynb")

## Select model
[back to top ⬆️](#Table-of-contents:)

Qwen3-VL Embedding Model list:

| Model Type | Models | Size | Layers | Sequence Length | Embedding Dimension | MRL Support | Instruction Aware |
|---|---|---|---|---|---|---|---|
| Multimodal Embedding | [Qwen3-VL-Embedding-2B](https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B) | 2B | 28 | 32K | 2048 | Yes | Yes |
| Multimodal Embedding | [Qwen3-VL-Embedding-8B](https://huggingface.co/Qwen/Qwen3-VL-Embedding-8B) | 8B | 36 | 32K | 4096 | Yes | Yes |

In [3]:
import ipywidgets as widgets

model_ids = ["Qwen/Qwen3-VL-Embedding-2B", "Qwen/Qwen3-VL-Embedding-8B"]

model_selector = widgets.Dropdown(
    options=model_ids,
    default=model_ids[0],
    description="Embedding Model:",
)

model_selector

Dropdown(description='Embedding Model:', options=('Qwen/Qwen3-VL-Embedding-2B', 'Qwen/Qwen3-VL-Embedding-8B'),…

## Convert model using Optimum Intel
[back to top ⬆️](#Table-of-contents:)

For convenience, we will use OpenVINO integration with HuggingFace Optimum. [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) is the interface between the Transformers and Diffusers libraries and the different tools and libraries provided by Intel to accelerate end-to-end pipelines on Intel architectures.

Among other use cases, Optimum Intel provides a simple interface to optimize your Transformers and Diffusers models, convert them to the OpenVINO Intermediate Representation (IR) format and run inference using OpenVINO Runtime. `optimum-cli` provides command line interface for model conversion and optimization.

General command format:

```bash
optimum-cli export openvino --model <model_id_or_path> --task <task> <output_dir>
```

where task is task to export the model for. Additionally, you can specify weights compression using `--weight-format` argument with one of following options: `fp32`, `fp16`, `int8` and `int4`.

In [4]:
to_compress = widgets.Checkbox(
    value=False,
    description="Weight compression",
    disabled=False,
)

visible_widgets = [to_compress]

options = widgets.VBox(visible_widgets)

options

The Qwen3-VL-Embedding model can be exported by `feature-extraction` task with Optimum-intel.

In [5]:
from pathlib import Path

model_id = model_selector.value

model_base_dir = Path(model_id.split("/")[-1])
additional_args = {"task": "feature-extraction"}

if to_compress.value:
    model_dir = model_base_dir / "INT8"
    additional_args.update({"weight-format": "int8"})
else:
    model_dir = model_base_dir / "FP16"
    additional_args.update({"weight-format": "fp16"})

In [6]:
from cmd_helper import optimum_cli

if not model_dir.exists():
    optimum_cli(model_id, model_dir, additional_args=additional_args)

**Export command:**

`optimum-cli export openvino --model Qwen/Qwen3-VL-Embedding-2B Qwen3-VL-Embedding-2B/FP16 --task feature-extraction --weight-format fp16`

## Run OpenVINO GenAI model inference
[back to top ⬆️](#Table-of-contents:)

[OpenVINO GenAI](https://github.com/openvinotoolkit/openvino.genai) provides an `EmbeddingPipeline` that runs the converted model end-to-end (tokenization, vision preprocessing, model inference and pooling) with a single call. Select device from dropdown list for running inference using OpenVINO.

In [7]:
from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU"])
device

Dropdown(description='Device:', options=('CPU', 'GPU', 'AUTO'), value='CPU')

The converted model can be loaded with the OpenVINO GenAI `EmbeddingPipeline`. Qwen3-VL-Embedding derives its embedding from the **last token** of the sequence, so we create the pipeline with `pooling_type=LAST_TOKEN` and `normalize=True` (L2 normalization, so a dot product between two embeddings equals their cosine similarity).

In [8]:
import openvino_genai

pipe = openvino_genai.EmbeddingPipeline(
    str(model_dir),
    device.value,
    pooling_type=openvino_genai.TextEmbeddingPipeline.PoolingType.LAST_TOKEN,
    normalize=True,
)

Model: OV Tokenizer
  NETWORK_NAME: tokenizer
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 16
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.5.0;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD RYZEN AI MAX+ PRO 395 w/ Radeon 8060S      
Model: OV Detokenizer
  NETWORK_NAME: detokenizer


Model: VLM vision embeddings merger model with VLSDPA optimization DISABLED
  NETWORK_NAME: Model6
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 16
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.5.0;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD RYZEN AI MAX+ PRO 395 w/ Radeon 8060S    

In [9]:
from io import BytesIO

import numpy as np
import openvino as ov
import requests
from PIL import Image

INSTRUCTION = "Represent the user's input."


def load_image(src):
    """Load an image from a URL or local path into an OpenVINO tensor."""
    if isinstance(src, str) and src.startswith("http"):
        image = Image.open(BytesIO(requests.get(src, timeout=30).content))
    else:
        image = Image.open(src)
    return ov.Tensor(np.array(image.convert("RGB"), dtype=np.uint8))


def get_embedding(pipe, inp, instruction=INSTRUCTION):
    """Get a normalized embedding for a single input (text, image, or text+image)."""
    text = ""
    images = []
    if isinstance(inp, dict):
        text = inp.get("text", "")
        if "image" in inp:
            images = [load_image(inp["image"])]
    elif isinstance(inp, str):
        text = inp

    if images:
        result = pipe.embed(text, embedding_prompt=instruction, images=images)
    else:
        result = pipe.embed(text, embedding_prompt=instruction)
    return np.asarray(result.embeddings.data, dtype=np.float32).reshape(1, -1)


# Example from https://hf-mirror.com/Qwen/Qwen3-VL-Embedding-2B#basic-usage-example
queries = [
    {"text": "A woman playing with her dog on a beach at sunset."},
    {"text": "Pet owner training dog outdoors near water."},
    {"text": "Woman surfing on waves during a sunny day."},
    {"text": "City skyline view from a high-rise building at night."},
]

documents = [
    {
        "text": "A woman shares a joyful moment with her golden retriever on a sun-drenched beach at sunset, as the dog offers its paw in a heartwarming display of companionship and trust."
    },
    {"image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"},
    {
        "text": "A woman shares a joyful moment with her golden retriever on a sun-drenched beach at sunset, as the dog offers its paw in a heartwarming display of companionship and trust.",
        "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg",
    },
]

# Inputs mix text, image, and text+image modalities so we compute embeddings one at a time.
all_inputs = queries + documents
embeddings = np.concatenate([get_embedding(pipe, inp) for inp in all_inputs], axis=0)

num_queries = len(queries)
query_embeddings = embeddings[:num_queries]
doc_embeddings = embeddings[num_queries:]

# Embeddings are L2-normalized, so a dot product equals the cosine similarity.
scores = query_embeddings @ doc_embeddings.T
print("Similarity scores (queries x documents):")
print(scores.tolist())

Similarity scores (queries x documents):
[[0.8345744609832764, 0.7781354784965515, 0.7880823612213135], [0.5348544120788574, 0.41248494386672974, 0.4257081151008606], [0.38840407133102417, 0.3267461955547333, 0.33486008644104004], [0.14878606796264648, 0.09109696000814438, 0.11166749894618988]]


### Compare with original PyTorch model (optional)
[back to top ⬆️](#Table-of-contents:)

Tick the checkbox below to also load the original PyTorch FP32 model and print the two score matrices side-by-side. The OpenVINO GenAI result should closely agree with the PyTorch reference (per-embedding cosine similarity above ~0.99, and differences of a few hundredths on this score matrix).

> **Note:** this cell is **disabled by default** because loading the full FP32 model requires ~8 GB of memory and adds a few minutes of runtime.

In [10]:
import ipywidgets as widgets

run_pt_compare_widget = widgets.Checkbox(
    value=False,
    description="Run PyTorch FP32 comparison",
    disabled=False,
)

run_pt_compare_widget

Checkbox(value=False, description='Run PyTorch FP32 comparison')

In [11]:
if not run_pt_compare_widget.value:
    print("Skipping PyTorch comparison. Tick the checkbox above and re-run this cell to enable.")
else:
    import gc
    import inspect

    import torch
    import torch.nn.functional as F
    from torch import Tensor
    from transformers import AutoModel, AutoProcessor
    from qwen_vl_utils import process_vision_info

    processor = AutoProcessor.from_pretrained(model_id)

    def last_token_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
        left_padding = attention_mask[:, -1].sum() == attention_mask.shape[0]
        if left_padding:
            return last_hidden_states[:, -1]
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]

    def get_embedding_pt(model, inp, instruction=INSTRUCTION):
        """PyTorch reference mirroring the GenAI ``get_embedding``. Strips processor keys the
        Qwen3VLModel forward does not accept (e.g. ``mm_token_type_ids``)."""
        content = []
        if isinstance(inp, dict):
            if "image" in inp:
                content.append({"type": "image", "image": inp["image"]})
            if "text" in inp:
                content.append({"type": "text", "text": inp["text"]})
        elif isinstance(inp, str):
            content.append({"type": "text", "text": inp})

        conversation = [
            {"role": "system", "content": [{"type": "text", "text": instruction}]},
            {"role": "user", "content": content},
        ]
        prompt = processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=False)
        image_inputs = process_vision_info(conversation)[0]
        if image_inputs:
            inputs = processor(text=[prompt], images=image_inputs, return_tensors="pt")
        else:
            inputs = processor(text=[prompt], return_tensors="pt")

        accepted = set(inspect.signature(model.forward).parameters)
        call_kwargs = {k: v for k, v in inputs.items() if k in accepted}

        with torch.no_grad():
            outputs = model(**call_kwargs)
        embedding = last_token_pool(outputs.last_hidden_state, inputs["attention_mask"])
        return F.normalize(embedding.to(torch.float32), p=2, dim=1)

    # AutoModel loads Qwen3VLModel (the backbone without LM head), matching what the
    # feature-extraction export wraps on the OpenVINO side.
    pt_model = AutoModel.from_pretrained(model_id, torch_dtype=torch.float32).eval()

    pt_embeddings = torch.cat([get_embedding_pt(pt_model, inp) for inp in all_inputs], dim=0)
    pt_scores = pt_embeddings[:num_queries] @ pt_embeddings[num_queries:].T
    pt_scores = pt_scores.cpu().numpy()

    print("PyTorch FP32 similarity scores:")
    print([[round(v, 4) for v in row] for row in pt_scores.tolist()])
    print("OpenVINO GenAI similarity scores:")
    print([[round(v, 4) for v in row] for row in scores.tolist()])

    max_abs_diff = np.abs(pt_scores - scores).max()
    print(f"Max |PT - OV| across the score matrix: {max_abs_diff:.4f}")

    del pt_model
    gc.collect()

Skipping PyTorch comparison. Tick the checkbox above and re-run this cell to enable.
